# HyDE
For a given query, HyDE retrieval pipeline contains 4 components:
1. Promptor: bulid prompt for generator based on specific task.
2. Generator: generates hypothesis documents using Large Language Model.
3. Encoder: encode hypothesis documents to HyDE vector.
4. Searcher: search nearest neighbour for the HyDE vector (dense retrieval).

### Initialize HyDE components
We use [pyserini](https://github.com/castorini/pyserini) as the search interface.

In [ ]:
import os
from pathlib import Path

DEFAULT_JAVA_HOME = Path('/home/iataghav/.local/java/jdk-21.0.11+10')
if not os.environ.get('JAVA_HOME') and DEFAULT_JAVA_HOME.is_dir():
    os.environ['JAVA_HOME'] = str(DEFAULT_JAVA_HOME)

if os.environ.get('JAVA_HOME'):
    java_bin = Path(os.environ['JAVA_HOME']) / 'bin'
    if java_bin.is_dir():
        os.environ['PATH'] = f"{java_bin}:" + os.environ.get('PATH', '')

print(f"JAVA_HOME={os.environ.get('JAVA_HOME', '<not set>')}")


In [ ]:
import json
import os
import sys
from pathlib import Path

def set_default_env(name, value):
    if not os.environ.get(name):
        os.environ[name] = str(value)

def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'src' / 'hyde').is_dir() and (path / 'setup.py').is_file():
            return path
    return start

ROOT_DIR = find_project_root()
os.chdir(ROOT_DIR)

src_path = str(ROOT_DIR / 'src')
set_default_env('PYTHONPATH', src_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

set_default_env('CUDA_VISIBLE_DEVICES', '4,5,6,7')
set_default_env('GLOBAL_VISIBLE_DEVICES', os.environ['CUDA_VISIBLE_DEVICES'])
set_default_env('TOKENIZERS_PARALLELISM', 'false')

DEFAULT_NOVELHOPQA_BOOKS_ROOT = ROOT_DIR / 'novelhopqa' / 'book-corpus-root'
FALLBACK_NOVELHOPQA_BOOKS_ROOT = Path('/home/iataghav/data/passing_meta_tag/novelhopqa/book-corpus-root')
current_books_root = os.environ.get('NOVELHOPQA_BOOKS_ROOT')
if not current_books_root or not (Path(current_books_root) / 'bookmeta.json').is_file():
    if (DEFAULT_NOVELHOPQA_BOOKS_ROOT / 'bookmeta.json').is_file():
        os.environ['NOVELHOPQA_BOOKS_ROOT'] = str(DEFAULT_NOVELHOPQA_BOOKS_ROOT)
    else:
        os.environ['NOVELHOPQA_BOOKS_ROOT'] = str(FALLBACK_NOVELHOPQA_BOOKS_ROOT)

HF_CACHE_ROOT = os.environ.get('SAADI_HF_CACHE_ROOT') or '/mnt/cache/taghavi'
set_default_env('HF_HOME', HF_CACHE_ROOT)
set_default_env('HF_HUB_CACHE', Path(os.environ['HF_HOME']) / 'hub')
set_default_env('HF_DATASETS_CACHE', Path(os.environ['HF_HOME']) / 'datasets')
set_default_env('TRANSFORMERS_CACHE', Path(os.environ['HF_HOME']) / 'transformers')

if os.environ.get('SAADI_HF_OFFLINE', '').lower() in {'1', 'true', 'yes', 'on'}:
    set_default_env('HF_HUB_OFFLINE', '1')
    set_default_env('TRANSFORMERS_OFFLINE', '1')
else:
    os.environ.pop('HF_HUB_OFFLINE', None)
    os.environ.pop('TRANSFORMERS_OFFLINE', None)

print(f'ROOT_DIR={ROOT_DIR}')
print(f'CUDA_VISIBLE_DEVICES={os.environ["CUDA_VISIBLE_DEVICES"]}')
print(f'HF_HOME={os.environ["HF_HOME"]}')

from pyserini.search.faiss import FaissSearcher
from pyserini.search.lucene import LuceneSearcher
from pyserini.encode import AutoQueryEncoder

import importlib
import hyde
import hyde.generator

importlib.invalidate_caches()
importlib.reload(hyde.generator)
importlib.reload(hyde)

from hyde import Promptor, TransformersGenerator, HyDE

if not hasattr(TransformersGenerator, 'prepare_snapshot'):
    raise RuntimeError('Loaded a stale hyde.TransformersGenerator. Restart the kernel and rerun from the first cell.')

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-30B-A3B-Instruct-2507'
promptor = Promptor('web search')
generator = TransformersGenerator(
    MODEL_NAME,
    api_key=os.getenv('HF_TOKEN'),
    n=8,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.8,
    device_map='auto',
    torch_dtype='auto',
    cache_dir=os.environ.get('HF_HUB_CACHE'),
    trust_remote_code=True,
    local_files_only=None,
)
qwen_snapshot = generator.prepare_snapshot()
print(f'Qwen snapshot={qwen_snapshot}')
encoder = AutoQueryEncoder(encoder_dir='facebook/contriever', pooling='mean')
searcher = FaissSearcher('contriever_msmarco_index/', encoder)
corpus = LuceneSearcher.from_prebuilt_index('msmarco-v1-passage')


### Build a HyDE pipeline

In [ ]:
hyde = HyDE(promptor, generator, encoder, searcher)

### Load example Query

In [ ]:
query = 'how long does it take to remove wisdom tooth'

### Build Zeroshot Prompt

In [ ]:
prompt = hyde.prompt(query)
print(prompt)

### Generate Hypothesis Documents

In [ ]:
hypothesis_documents = hyde.generate(query)
for i, doc in enumerate(hypothesis_documents):
    print(f'HyDE Generated Document: {i}')
    print(doc.strip())

### Encode HyDE vector

In [ ]:
hyde_vector = hyde.encode(query, hypothesis_documents)
print(hyde_vector.shape)

### Search Relevant Documents

In [ ]:
hits = hyde.search(hyde_vector, k=10)
for i, hit in enumerate(hits):
    print(f'HyDE Retrieved Document: {i}')
    print(hit.docid)
    print(json.loads(corpus.doc(hit.docid).raw())['contents'])

### End to End Search

e2e search will directly go through all the steps descripted above.

In [ ]:
hits = hyde.e2e_search(query, k=10)
for i, hit in enumerate(hits):
    print(f'HyDE Retrieved Document: {i}')
    print(hit.docid)
    print(json.loads(corpus.doc(hit.docid).raw())['contents'])